In [9]:
public interface IInstallable
{
    void Install();
}

public abstract class Component
{
    public string Name { get; set; }
    public string Manufacturer { get; set; }
    public decimal Price { get; set; }

    protected Component(string name, string manufacturer, decimal price)
    {
        Name = name;
        Manufacturer = manufacturer;
        Price = price;
    }

    public virtual string GetInfo()
    {
        return $"{Name} ({Manufacturer}), цена: {Price} руб.";
    }

    public override string ToString()
    {
        return GetInfo();
    }
}

public class CPU : Component, IInstallable
{
    public int Cores { get; set; }
    public double FrequencyGHz { get; set; }

    public CPU(string name, string manufacturer, decimal price, int cores, double frequencyGHz)
        : base(name, manufacturer, price)
    {
        Cores = cores;
        FrequencyGHz = frequencyGHz;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", ядер: {Cores}, частота: {FrequencyGHz} ГГц";
    }

    public void Install()
    {
        Console.WriteLine($"Процессор {Name} установлен в сокет.");
    }

    public static bool operator ==(CPU a, CPU b)
    {   
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.Cores == b.Cores && a.FrequencyGHz == b.FrequencyGHz;
    }

    public static bool operator !=(CPU a, CPU b)
    {
        return !(a == b);
    }

    public override bool Equals(object obj) => obj is CPU other && this == other;
    public override int GetHashCode() => HashCode.Combine(Name, Cores, FrequencyGHz);
}

public class RAM : Component, IInstallable
{
    public int SizeGB { get; set; }
    public string Type { get; set; } // например, DDR4

    public RAM(string name, string manufacturer, decimal price, int sizeGB, string type)
        : base(name, manufacturer, price)
    {
        SizeGB = sizeGB;
        Type = type;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", объём: {SizeGB} ГБ, тип: {Type}";
    }

    public void Install()
    {
        Console.WriteLine($"Память {Name} вставлена в слот DIMM.");
    }

    public static bool operator ==(RAM a, RAM b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.SizeGB == b.SizeGB && a.Type == b.Type;
    }

    public static bool operator !=(RAM a, RAM b)
    {
        return !(a == b);
    }

    public override bool Equals(object obj) => obj is RAM other && this == other;
    public override int GetHashCode() => HashCode.Combine(Name, SizeGB, Type);
}

public class GPU : Component, IInstallable
{
    public int VideoMemoryGB { get; set; }

    public GPU(string name, string manufacturer, decimal price, int videoMemoryGB)
        : base(name, manufacturer, price)
    {
        VideoMemoryGB = videoMemoryGB;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", видеопамять: {VideoMemoryGB} ГБ";
    }

    public void Install()
    {
        Console.WriteLine($"Видеокарта {Name} установлена в слот PCI-E.");
    }

    public static bool operator ==(GPU a, GPU b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.VideoMemoryGB == b.VideoMemoryGB;
    }

    public static bool operator !=(GPU a, GPU b)
    {
        return !(a == b);
    }

    public override bool Equals(object obj) => obj is GPU other && this == other;
    public override int GetHashCode() => HashCode.Combine(Name, VideoMemoryGB);
}

public class HDD : Component, IInstallable
{
    public int CapacityGB { get; set; }

    public HDD(string name, string manufacturer, decimal price, int capacityGB)
        : base(name, manufacturer, price)
    {
        CapacityGB = capacityGB;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", ёмкость: {CapacityGB} ГБ";
    }

    public void Install()
    {
        Console.WriteLine($"Жёсткий диск {Name} подключен к SATA.");
    }

    public static bool operator ==(HDD a, HDD b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.CapacityGB == b.CapacityGB;
    }

    public static bool operator !=(HDD a, HDD b)
    {
        return !(a == b);
    }

    public override bool Equals(object obj) => obj is HDD other && this == other;
    public override int GetHashCode() => HashCode.Combine(Name, CapacityGB);
}

public class Computer
{
    public string Name { get; set; }

    private List<Component> components = new List<Component>();

    public Computer(string name)
    {
        Name = name;
    }

    public void AddComponent<T>(T component) where T : Component, IInstallable
    {
        component.Install();
        components.Add(component);
    }

    public bool RemoveComponent<T>(T component) where T : Component
    {
        bool removed = components.Remove(component);
        if (removed)
            Console.WriteLine($"Компонент {component.Name} удалён.");
        else
            Console.WriteLine($"Компонент {component.Name} не найден в компьютере.");
        return removed;
    }

    public decimal GetTotalPrice()
    {
        decimal total = 0;
        foreach (Component c in components)
        {
            total += c.Price;
        }
        return total;
    }

    public void ShowInfo()
    {
        Console.WriteLine($"=== Компьютер «{Name}» ===");
        foreach (Component c in components)
        {
            Console.WriteLine(" - " + c); // вызовется ToString()
        }
        Console.WriteLine($"Итого: {GetTotalPrice()} руб.");
    }
}

public static class ComputerFactory
{
    public static Computer CreateCustom<TCpu, TRam, THdd>(string name, TCpu cpu, TRam ram, THdd hdd)
        where TCpu : Component, IInstallable
        where TRam : Component, IInstallable
        where THdd : Component, IInstallable
    {
        Computer computer = new Computer(name);
        computer.AddComponent(cpu);
        computer.AddComponent(ram);
        computer.AddComponent(hdd);
        return computer;
    }

    public static Computer CreateOffice()
    {
        return CreateCustom(
            "Офисный",
            new CPU("Core i3-12100", "Intel", 9000, 4, 3.3),
            new RAM("Kingston Fury 8GB", "Kingston", 2500, 8, "DDR4"),
            new HDD("Barracuda 1TB", "Seagate", 3500, 1000));
    }

    public static Computer CreateGaming()
    {
        Computer computer = CreateCustom(
            "Игровой",
            new CPU("Ryzen 7 5800X", "AMD", 22000, 8, 3.8),
            new RAM("Corsair Vengeance 32GB", "Corsair", 8000, 32, "DDR4"),
            new HDD("WD Blue 2TB", "Western Digital", 5000, 2000));

        computer.AddComponent(new GPU("GeForce RTX 4070", "NVIDIA", 55000, 12));
        return computer;
    }
}

public class Program
{
    public static void Main()
    {
        Console.WriteLine("--- 1. Сборка компьютера вручную ---");
        Computer pc = new Computer("Мой ПК");
        CPU cpu = new CPU("Core i5-12400", "Intel", 15000, 6, 2.5);
        RAM ram = new RAM("Crucial 16GB", "Crucial", 4000, 16, "DDR4");
        GPU gpu = new GPU("GeForce GTX 1660", "NVIDIA", 20000, 6);
        HDD hdd = new HDD("Toshiba P300 1TB", "Toshiba", 3000, 1000);

        pc.AddComponent(cpu);
        pc.AddComponent(ram);
        pc.AddComponent(gpu);
        pc.AddComponent(hdd);

        Console.WriteLine();
        pc.ShowInfo();

        Console.WriteLine("\n--- 2. Полиморфизм: GetInfo() у разных компонентов ---");
        List<Component> parts = new List<Component> { cpu, ram, gpu, hdd };
        foreach (Component part in parts)
        {
            Console.WriteLine(part.GetInfo());
        }

        Console.WriteLine("\n--- 3. Сравнение компонентов (== и !=) ---");
        CPU sameCpu = new CPU("Core i5-12400", "Intel", 15000, 6, 2.5);
        CPU otherCpu = new CPU("Ryzen 5 5600", "AMD", 14000, 6, 3.5);
        Console.WriteLine($"cpu == sameCpu: {cpu == sameCpu}");
        Console.WriteLine($"cpu == otherCpu: {cpu == otherCpu}");
        Console.WriteLine($"cpu != otherCpu: {cpu != otherCpu}");

        Console.WriteLine("\n--- 4. Удаление компонента ---");
        pc.RemoveComponent(gpu);
        pc.RemoveComponent(gpu); // второй раз — уже нет
        Console.WriteLine($"Стоимость после удаления: {pc.GetTotalPrice()} руб.");

        Console.WriteLine("\n--- 5. Фабрика компьютеров ---");
        Computer office = ComputerFactory.CreateOffice();
        Console.WriteLine();
        office.ShowInfo();

        Console.WriteLine();
        Computer gaming = ComputerFactory.CreateGaming();
        Console.WriteLine();
        gaming.ShowInfo();

    }
}
Program.Main();

--- 1. Сборка компьютера вручную ---
Процессор Core i5-12400 установлен в сокет.
Память Crucial 16GB вставлена в слот DIMM.
Видеокарта GeForce GTX 1660 установлена в слот PCI-E.
Жёсткий диск Toshiba P300 1TB подключен к SATA.

=== Компьютер «Мой ПК» ===
 - Core i5-12400 (Intel), цена: 15000 руб., ядер: 6, частота: 2.5 ГГц
 - Crucial 16GB (Crucial), цена: 4000 руб., объём: 16 ГБ, тип: DDR4
 - GeForce GTX 1660 (NVIDIA), цена: 20000 руб., видеопамять: 6 ГБ
 - Toshiba P300 1TB (Toshiba), цена: 3000 руб., ёмкость: 1000 ГБ
Итого: 42000 руб.

--- 2. Полиморфизм: GetInfo() у разных компонентов ---
Core i5-12400 (Intel), цена: 15000 руб., ядер: 6, частота: 2.5 ГГц
Crucial 16GB (Crucial), цена: 4000 руб., объём: 16 ГБ, тип: DDR4
GeForce GTX 1660 (NVIDIA), цена: 20000 руб., видеопамять: 6 ГБ
Toshiba P300 1TB (Toshiba), цена: 3000 руб., ёмкость: 1000 ГБ

--- 3. Сравнение компонентов (== и !=) ---
cpu == sameCpu: True
cpu == otherCpu: False
cpu != otherCpu: True

--- 4. Удаление компонента ---
Комп